# To prove true campaign lift, we need to compare your Control group (No Promo) against your Test groups (Promo 1 and Promo 2) across three specific dimensions:

## Total Order Volume: Did the promo drive a massive spike in traffic?

## Average Order Value (AOV): Did the discount lower our average ticket size, or did it encourage them to spend more overall?

## Average Basket Size: Did they buy more items per transaction?

### 1. The Pull: Connect to DuckD and pull fct_orders table.

In [2]:
import pandas as pd
from connect import get_warehouse_connection

con = get_warehouse_connection('../mean_mug_analytics/mean_mug.duckdb')
df = con.execute('SELECT * FROM fct_orders').df()

Successfully connected to warehouse: ../mean_mug_analytics/mean_mug.duckdb


In [3]:
df.head()

,order_id,customer_id,store_location_id,promo_id,order_date,order_time,payment_type_lower,total_amount,total_items_in_basket
0,1,1,1,<NA>,2024-01-26,09:21:00,app,7.50,2
1,2,1,2,2,2024-02-12,07:54:00,app,4.25,1
2,3,2,2,2,2024-02-08,09:22:00,app,16.00,4
3,4,2,1,<NA>,2024-03-26,17:06:00,card,4.25,1
4,5,3,2,<NA>,2024-03-12,13:19:00,cash,10.00,3


### 2. The Clean Up: Right now, the control group has a promo_id of <NA>. Pandas .groupby() automatically drops null values by default! 

In [4]:
# cleaning with .fillna()
# The column is integer, if we want to add a string we need to convert the dtype to string:
df['promo_id'] = df['promo_id'].astype('string')
# Fill NA:
df['promo_id'] = df['promo_id'].fillna('No_Promo')
df.head()

,order_id,customer_id,store_location_id,promo_id,order_date,order_time,payment_type_lower,total_amount,total_items_in_basket
0,1,1,1,No_Promo,2024-01-26,09:21:00,app,7.50,2
1,2,1,2,2,2024-02-12,07:54:00,app,4.25,1
2,3,2,2,2,2024-02-08,09:22:00,app,16.00,4
3,4,2,1,No_Promo,2024-03-26,17:06:00,card,4.25,1
4,5,3,2,No_Promo,2024-03-12,13:19:00,cash,10.00,3


### The Aggregation: Group the dataframe by promo_id.

In [5]:
df_lift = df.groupby('promo_id').agg(
    total_vol = ('order_id','nunique'),
    aov = ('total_amount', 'mean'),
    basket_density = ('total_items_in_basket', 'mean')
).reset_index()
df_lift

,promo_id,total_vol,aov,basket_density
0,1,680,11.369853,2.531664
1,2,730,10.921575,2.449315
2,No_Promo,1414,10.957037,2.438472


In [6]:
import importlib
import bigquery_export

# Force Jupyter to read the updated bigquery_export.py file
importlib.reload(bigquery_export)

# Updated function
bigquery_export.push_to_warehouse(df_lift, "mart_campaign_lift")

Uploading mart_campaign_lift to BigQuery...
Success: mart_campaign_lift is live!


In [7]:
con.close()